# Spatial Mapping Analysis

---

## Overview
Spatially mapping all indices in `indice_summary.csv` with water quality data in `upper_keys_wq.geoJSON` for spatial patterns and trends over time with overlay using the Rasterio library. Preprocessed `clipped_manifest` with each scenes `.nc` is required for proper display of Sentinel-2 imagery.

1. Base map function 
2. User input plot function
3. User call function
4. Seagrass Dynamics Upper Keys FL for 2015-2024 Animation

This analysis was attempted to be created for easy reusability.

---

In [ ]:
# pip install..
!pip install Cartopy

## Imports

In [ ]:
import os
import cartopy.crs as ccrs
import cartopy.feature as cf
import pandas as pd
import geopandas as gpd 
import rioxarray as rxr 
import rasterio as rio 
import xarray as xr 
import matplotlib
import matplotlib.pyplot as plt 
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np 

### Load Data

In [ ]:
aoi_shapefile = "seagrass_project/study_area/UpperKeys_FL.shp"
clipped_manifest = pd.read_csv("processed/clipped_manifest.csv", parse_dates=["date"])
clipped_manifest["tile"] = clipped_manifest["scene_id"].str.split("_").str[0]
indices_summary = pd.read_csv("processed/indices_summary.csv", parse_dates=["time"])
wq_gdf = gpd.read_file("seagrass_project/wq_data/upper_keys_wq.geojson")

In [ ]:
wq_gdf.to_crs("EPSG:32617")
print(wq_gdf.crs)
print(wq_gdf.head())

## Base Map Function

Wraps Cartopy code into a function to make quick basemaps - projection set up is Mercator, draws coastline, maps gridlines, and zooms to extent of Upper Keys, FL. Easy reuse for any plot that needs a basemap by calling make_basemap(ax).

In [ ]:
def make_basemap(ax, title="Upper Keys FL"):
    """"
    Creates a basic Cartopy basemap with coastline of the Upper Keys AOI.
    Raster overlay shows vegetation by itself.
    """
    crs = ccrs.PlateCarree()
    projection = ccrs.Mercator()
    gl = ax.gridlines(crs=crs, draw_labels=True,
                      linewidth=.6, color='grey', alpha=0.5, linestyle='-.')
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}
    ax.add_feature(cf.COASTLINE.with_scale("10m"), lw=0.8, zorder=4)

    lon_min = -80.622
    lon_max = -80.136
    lat_min = 24.849
    lat_max = 25.374
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=crs)
    ax.set_title(title)

## User Input Plot Function
Takes a scene ID, index name, and WQ parameter as inputs. Opens that scenes netCDF, calculates the chosen index as a 2D raster, reprojects it to lat/lon so it aligns with the map projection, and filters WQ measurements within 30 days of scene date. 

In [ ]:
def plot_scene(scene_id, index="NDAVI", wq_param="Turbidity (NTU)"):
    """"Plots the a chosen scene's index raster as a semitransparent layer over basemap,
     with overlaying WQ stations as scatter points colored by wq_params.
    """

    row = clipped_manifest[clipped_manifest["scene_id"] == scene_id].iloc[0]
    ds = xr.open_dataset(row["clipped_path"])

    if index == "NDAVI":
        raster = (ds.nir - ds.blue) / (ds.nir + ds.blue)
    elif index == "NDVI":
        raster = (ds.nir - ds.red) / (ds.nir + ds.red)
    elif index == "NDTI":
        raster = (ds.red - ds.green) / (ds.red + ds.green)
    elif index == "NDWI": 
        raster = (ds.green - ds.nir) / (ds.green + ds.nir)
    elif index == "SSSII":
        raster = (ds.green - ds.red) / (ds.green + ds.red)
    elif index == "DII":
        raster = np.log(ds.green.where(ds.green > 0)) - np.log(ds.blue.where(ds.blue > 0))
    else:
        raise ValueError(f"Unknown index: {index}") 
    
    # get bounds in lat/lon
    ds_4326 = ds.rio.reproject("EPSG:4326")
    left, bottom, right, top = ds_4326.rio.bounds()
    raster_4326 = raster.rio.reproject("EPSG:4326")

    # filter WQ to scene date +- 30 days
    scene_date = pd.to_datetime(row["date"])
    wq_near = wq_gdf[
        (pd.to_datetime(wq_gdf["Date"]) >= scene_date - pd.Timedelta("30D")) &
        (pd.to_datetime(wq_gdf["Date"]) <= scene_date + pd.Timedelta("30D"))
    ].to_crs("EPSG:4326")
    
    # create figure and axes 
    fig = plt.figure(dpi=150, figsize=(10, 8))
    ax = plt.axes(projection=ccrs.Mercator())
    make_basemap(ax, title=f"{index} - {scene_id}") 

    # overlay index raster
    ax.imshow(raster_4326.values, origin="upper",
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        cmap="YlGn", 
        alpha=0.9, 
        vmin=-0.3, 
        vmax=0.6
    )

    # overlay WQ stations
    if not wq_near.empty and wq_param in wq_near.columns:
        sc = ax.scatter(wq_near.geometry.x, wq_near.geometry.y,
                        c=wq_near[wq_param], cmap="Reds",
                        s=60, zorder=6, transform=ccrs.PlateCarree(),
                        edgecolors="black", linewidths=0.5)
        fig.colorbar(sc, ax=ax, label=wq_param, shrink=0.6)

        plt.tight_layout()
        plt.show()
        ds.close()

## Call functions 
The user function is designed for easy reuse and full freedom of the scene and water quality parameter desired for analysis to produce a map by scene_id (including date), WQ, and index. For the analysis of Seagrass dynamics in the Upper Keys, NDAVI with Turbidity, and DII with Secchi Depth (m), were chosen for comparison. Based on the findings in the time-series, the functions allow any comparison visual output to be created and saved to support or revute findings of interest in the final report.

In [ ]:
scene_id = "T17RNH_20151113T160052"
index = "NDTI"
wq_param = "Turbidity (NTU)"

plot_scene(scene_id, index, wq_param)

plt.savefig(f"seagrass_project/Turbidity_{index}_{scene_id}.png")

### Spatial trends

## Upper Keys Seagrass Dynamics 2015-2024 Animation
Utilizing `FuncAnimation(fig, update, frames=N interval=600)` through `matplotlib.animation` to use the user function to open each scene and create an animated time-series of NDAVI for visualizing change in turbidity overtime. IPython function `ani.to_jshtml()` converts the animation interactive HTML. Raster scene files are large so with Matplotlib setting the frame limit with `rcParams` to 50 allowed for no dropping of frames out of 88.

In [ ]:
tile_scenes = clipped_manifest[clipped_manifest["tile"] == "T17RNH"].sort_values("date")

fig = plt.figure(dpi=120, figsize=(10, 8))
ax = plt.axes(projection=ccrs.Mercator())
make_basemap(ax, title="NDAVI Upper Keys FL 2015-2024")
img_artist = [None]

def update(i):
    row   = tile_scenes.iloc[i]
    ds    = xr.open_dataset(row["clipped_path"])
    ndavi  = (ds.nir - ds.blue) / (ds.nir + ds.blue)
    r4326 = ndavi.rio.reproject("EPSG:4326")

    # downsample to reduce memory — every 4th pixel
    r_down = r4326.values[::4, ::4]
    left, bottom, right, top = r4326.rio.bounds()

    if img_artist[0] is not None:
        try:
            img_artist[0].remove()
        except ValueError:
            pass

    img_artist[0] = ax.imshow(
        r_down,
        origin="upper",
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        cmap="YlOrRd",
        alpha=0.85,
        vmin=-0.3, vmax=0.6,
        zorder=2
    )
    ax.set_title(f"NDAVI — {row['date'].date()}")
    ds.close()

matplotlib.rcParams['animation.embed_limit'] = 50  # MB limit, increase as needed

ani = FuncAnimation(fig, update, frames=len(tile_scenes), interval=600)

ani.save("seagrass_project/ndti_animation_2015_2024.gif", writer="pillow", fps=2, dpi=80)
print("Saved animation.")

HTML(ani.to_jshtml(fps=2, embed_frames=True))

---

## Summary
**Ouputs:**
- Predefined user input functions    `make_basemap` and `plot_scenes` to visualize any vegetation indices overlayed with a WQ parameter.
- Animated visual of Seagrass Dynamics in Upper Keys FL for the study period 2015-2024.

**Next:** Computational time-series analysis of integrated Sentinel-2 indice data and water quality data for the study area.

## Resources and references
- nilbarde. Ans. (2022, Jan 22). How to save output from cells in jupyter notebook. [Stack Overflow](https://stackoverflow.com/questions/72811926/anybody-know-how-to-save-image-output-from-cells-in-jupyter-notebook)
- Norris, W. (2021, Jun 25). [*Visualizing Satellite Data Using Matplotlib and Cartopy.*](https://towardsdatascience.com/visualizing-satellite-data-using-matplotlib-and-cartopy-8274acb07b84/) Towards Data Science.
- Franko, L. (2022, Jun 28). [*Climate Data Visualization with Python.*](https://medium.com/@lubomirfranko/climate-data-visualisation-with-python-visualise-climate-data-using-cartopy-and-xarray-cf35a60ca8ee) Medium.


